Loads the preprocessed AnnDatas and guidance graph from tutorial 1 and
runs :func:`storm.models.fit_STORM` with
:class:`storm.models.PairedSTORMModel` to produce a ``.dill`` checkpoint.

STORM's training objective combines a multi-modality, multi-timepoint
ELBO with a graph-guidance ELBO, a masked-ℓ1 sparsity penalty on the
decoder weights, a spatial-adjacency reconstruction loss, a paired
cross-modality consistency loss, and a two-headed adversarial
discriminator (one head over modality, one over timepoint). All terms
share a single set of hyperparameters in ``compile_kws``; this tutorial
walks through what each one controls and why the default values are
chosen the way they are.

In a tutorial setting we run only a handful of fine-tune epochs so the
cell finishes in minutes — the resulting embedding is intentionally
noisy. The **downstream notebooks (3 and 4) are designed to be re-run
against any ``.dill`` checkpoint**, so once you have a full-length
training run, just point ``TRAINED_DILL`` in those notebooks at your
final model and rerun.

# STORM tutorial 2: training

This notebook trains a paired RNA + ATAC STORM model on the artifacts
from [tutorial 1 (preprocessing)](./tutorial_1_preprocess.ipynb).

## What ``fit_STORM`` is actually doing

:func:`storm.models.fit_STORM` runs a **three-stage loop** that matches
the manuscript's training schedule (Chen et al., 2026, Methods —
*Implementation details and weight freezing*):

1. **Pretrain** a paired model with the user-supplied ``compile_kws``.
   This stage further uses a *staged weight-freezing schedule* to
   stabilise optimisation:
   * train the RNA encoder/decoder for ``n_epochs_rna`` (default 50)
     epochs with everything else frozen;
   * then unfreeze the ATAC encoder/decoder and the discriminator,
     and train for ``n_epochs_atac`` (default 100) epochs with the RNA
     parameters frozen;
   * finally, unfreeze everything and train jointly until the early
     stopping criterion fires.
2. **Estimate balancing weights** from the pretrain embedding. We cluster
   ``u`` per (modality, timepoint) group via Leiden, then weight each
   spot by how well its cluster matches a cluster in another group via
   a contrast-sharpened cosine similarity (``f(c) = c⁴`` if ``c > 0.5``,
   else ``0``). The resulting weights re-normalise the discriminator's
   cross-entropy so that rare-but-shared populations are not drowned out
   by abundant ones during alignment.
3. **Fine-tune** the model with the balancing weights applied, gradually
   annealing the Gaussian noise injected into the discriminator's
   inputs to zero. The fine-tune stage overlays ``finetune_compile_kws``
   on top of ``compile_kws`` — see :func:`storm.models.fit_STORM` for the
   defaults.

## The full training objective

The combined objective is (schematically):

$$
\min_\psi\ \lambda_{D^{(m)}}\,L_{D}^{(m)} + \lambda_{D^{(t)}}\,L_{D}^{(t)}
$$
$$
\max_{\theta,\phi}\ \big[\,\lambda_{D^{(m)}}\,L_{D}^{(m)} + \lambda_{D^{(t)}}\,L_{D}^{(t)}\big]
+ \lambda_{G_f}\,L_{G_f} + \lambda_m \sum_{m,t} L_{m,t}
- \lambda_{\ell_1}(L_{\ell_1}(W) + L_{\ell_1}(W'))
- \lambda_{G_\mathrm{spatial}}\,L_{G_\mathrm{spatial}}
- \lambda_\mathrm{MSE}\,L_\mathrm{MSE}.
$$

Read this as: the discriminator is trained to *minimise* its
modality/timepoint classification loss; the encoder + decoders are
trained adversarially to *maximise* it (so the cell embedding cannot be
linearly separated by modality or timepoint), while also maximising the
data + graph ELBO and minimising the three structural penalties
(``ℓ₁``, spatial adjacency, paired consistency). RMSprop with no
momentum is the default optimiser.

The ``λ`` symbols above map directly onto the ``lam_*`` keys in
``compile_kws`` — section 5 below walks through each one.

In [1]:
from itertools import chain
import os
import sys

_REPO_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), os.pardir)) \
    if "__file__" in globals() else os.path.abspath(os.path.join(os.getcwd(), os.pardir))
for _candidate in (_REPO_ROOT, os.getcwd()):
    if os.path.isdir(os.path.join(_candidate, "storm")) and _candidate not in sys.path:
        sys.path.insert(0, _candidate)
        break

import anndata as ad
import networkx as nx

import storm

storm.plot.set_publication_params()

## 1. Parameters and paths

``MAX_EPOCHS = 5`` keeps the demo short; raise it to ~190 for a full
reproduction. The remaining schedule knobs should scale with it. For the
published two-timepoint mouse-brain run we used
``MAX_EPOCHS = 190``, ``ALIGN_BURNIN = 32``,
``PATIENCE = 16``, ``REDUCE_LR_PATIENCE = 8``.

What each one controls:

* **``ALIGN_BURNIN``** — number of epochs of pure encoder/decoder
  warm-up at the start of fine-tuning before the adversarial alignment
  pressure is turned on. Lets the embedding settle into a sensible
  geometry before the discriminator starts pulling on it.
* **``PATIENCE``** — early-stopping patience on validation loss
  (10% held out by default). Training stops once validation has not
  improved for this many epochs.
* **``REDUCE_LR_PATIENCE``** — patience for the LR scheduler. After
  this many non-improving epochs the learning rate is divided by 10.

The manuscript scales these automatically with dataset size and learning
rate to match training progress equivalent to 1,000 (LR decay), 2,000
(early stop), and 16,000 (max) iterations at ``lr = 0.002``; the values
below are the manually tuned versions used for the two-timepoint
P21–P22 dataset.

In [2]:
MAX_EPOCHS = 5
ALIGN_BURNIN = 2
PATIENCE = 4
REDUCE_LR_PATIENCE = 2

TUTORIAL_OUT = "../artifacts/storm_tutorial"
PREP_DIR = f"{TUTORIAL_OUT}/preprocessed"
MODEL_DIR = f"{TUTORIAL_OUT}/model"
os.makedirs(MODEL_DIR, exist_ok=True)

# Inputs (from tutorial_1_preprocess.ipynb).
PREP_RNA = f"{PREP_DIR}/rna_preprocessed.h5ad"
PREP_ATAC = f"{PREP_DIR}/atac_preprocessed.h5ad"
PREP_GRAPH = f"{PREP_DIR}/guidance.graphml.gz"

# Output (consumed by tutorials 3 and 4).
TUTORIAL_DILL = "../dill_files/storm_tutorial.dill"
os.makedirs(os.path.dirname(TUTORIAL_DILL), exist_ok=True)

## 2. Load the preprocessed artifacts

These are exactly the three files written at the end of tutorial 1.

In [3]:
rna = ad.read_h5ad(PREP_RNA)
atac = ad.read_h5ad(PREP_ATAC)
guidance = nx.read_graphml(PREP_GRAPH)

print("RNA:", rna)
print("ATAC:", atac)
print(f"Guidance graph: {guidance.number_of_nodes()} nodes, "
      f"{guidance.number_of_edges()} edges")

RNA: AnnData object with n_obs × n_vars = 11324 × 2871
    obs: 'timepoint', 'RNA_clusters', 'ATAC_clusters'
    var: 'n_cells', 'highly_variable', 'mean', 'std', 'chrom', 'chromStart', 'chromEnd', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'blockCount', 'blockSizes', 'blockStarts', 'gene_id', 'gene_type'
    uns: 'log1p', 'moranI', 'pca', 'spatial_neighbors', 'storm_genes_idx', 'storm_gp_names', 'storm_source_genes_idx', 'storm_sources_categories_label_encoder', 'storm_target_genes_idx', 'storm_targets_categories_label_encoder', 'terms'
    obsm: 'X_pca', 'X_spa', 'spatial', 'spatial_adj'
    varm: 'I', 'PCs', 'storm_gene_peaks', 'storm_gp_sources', 'storm_gp_sources_categories', 'storm_gp_targets', 'storm_gp_targets_categories'
    layers: 'counts'
    obsp: 'spatial_connectivities', 'spatial_distances'
ATAC: AnnData object with n_obs × n_vars = 11324 × 4058
    obs: 'orig.ident', 'nCount_originalexp', 'nFeature_originalexp', 'BlacklistRatio', 'nDiFrags', 'nFrags

## 3. Configure the per-modality dataset

:func:`storm.models.configure_dataset` records the layout in
``adata.uns[storm.config.ANNDATA_KEY]`` — which probabilistic decoder to
use, which layer holds raw counts, which ``obsm`` slot is the encoder
input, and which ``.obs`` columns to read for batch / timepoint /
cell-type information.

The choices below match the published STORM setup:

| Argument | Value | Why |
|----------|-------|-----|
| ``prob_model`` | ``"NB"`` | Negative-binomial decoder over raw counts. STORM models both RNA counts and ATAC peak counts as NB — captures the count nature and overdispersion. |
| ``use_layer="counts"`` | raw counts in ``.layers`` | Decoders are always trained on the *original* count space. Spatial smoothing / PCA only affects the encoder input. |
| ``use_rep="X_pca"`` / ``"X_lsi"`` | PCA / LSI representation | First-layer linear preprocessing inside the encoder (manuscript: "linear preprocessing step as the first layer"). |
| ``use_batch="timepoint"`` | timepoint column | Lets the discriminator treat timepoint as a soft batch effect for the alignment loss. |
| ``use_timepoint="timepoint"`` | timepoint column | Activates the **timepoint head** of the discriminator (the temporal-alignment objective added on top of vanilla GLUE). |
| ``use_obs_names=True`` | — | Pair RNA and ATAC spots by matching ``obs_names`` so the paired MSE consistency loss can fire. |

In [4]:
storm.models.configure_dataset(
    rna, "NB",
    use_highly_variable=True,
    use_layer="counts",
    use_rep="X_pca",
    use_obs_names=True,
    use_batch="timepoint",
    use_timepoint="timepoint",
)
storm.models.configure_dataset(
    atac, "NB",
    use_highly_variable=True,
    use_rep="X_lsi",
    use_obs_names=True,
    use_batch="timepoint",
    use_timepoint="timepoint",
)

## 4. Subset the guidance graph to highly-variable features

The encoder only sees HVF columns; guidance edges to dropped features
would be wasted compute (and would inflate the graph likelihood for
features the data decoder cannot reconstruct).

In [5]:
guidance_hvf = guidance.subgraph(
    chain(
        rna.var.query("highly_variable").index,
        atac.var.query("highly_variable").index,
    )
).copy()
print(f"HVF guidance: {guidance_hvf.number_of_nodes()} nodes, "
      f"{guidance_hvf.number_of_edges()} edges")

HVF guidance: 6929 nodes, 15069 edges


## 5. Fit the paired STORM model

Each entry in ``compile_kws`` corresponds to one term in the training
objective from section 0:

| Key | Symbol | Role |
|-----|--------|------|
| ``modality_weight={"rna": 1.0, "atac": 3.0}`` | scales each ``L_{m,t}`` | Per-modality reconstruction weight. ATAC is sparser and harder to fit than RNA, so it is upweighted to keep both modalities contributing comparable gradient signal. |
| ``lr=0.002`` | — | RMSprop learning rate (no momentum), per the manuscript. |
| ``lam_cos=100.0`` | ``λ_MSE`` | Cross-modality paired consistency. MSE between the cell embedding inferred from RNA and the one inferred from ATAC for the same spot. Pushes ``q(u|x_RNA) ≈ q(u|x_ATAC)`` so the two modalities share one embedding space. |
| ``lam_adj=1000.0`` | ``λ_{G_spatial}`` | Spatial adjacency reconstruction. BCE over positive (neighbouring) and sampled negative (non-neighbouring) spot pairs in the program-activity embeddings ``z_m(u)``. Local spatial coherence. |
| ``lam_masked_l1=0.1`` | ``λ_{ℓ1}`` | Masked ℓ₁ sparsity on decoder weights. Within each program, drives most feature loadings to zero so the remaining nonzero ones become high-confidence drivers. |
| ``lam_align=1000.0`` | ``λ_{D^{(m)}}`` | Modality discriminator weight. Strength of the adversarial signal that aligns RNA and ATAC in the cell embedding ``u``. |
| ``lam_tmp=10.0`` | ``λ_{D^{(t)}}`` | Timepoint discriminator weight. Same idea but across P21 vs. P22. Deliberately ~100× smaller than ``lam_align`` — the goal is *selective invariance*, aligning enough to share an embedding but not so much that genuine developmental change is erased (Problem 1 in tutorial 1). |

And on the ``fit_kws`` side:

* ``directory`` — where TensorBoard logs and intermediate checkpoints
  are written. ``fit_STORM`` further appends ``/pretrain`` and
  ``/fine-tune`` subdirectories.
* ``max_epochs`` / ``align_burnin`` / ``safe_burnin`` / ``patience`` /
  ``reduce_lr_patience`` — schedule knobs, explained in section 1.
* ``safe_burnin=True`` injects Gaussian noise (``Σ = 1.5 ×`` empirical
  covariance of ``u``) into the discriminator inputs during burn-in to
  stabilise the early adversarial signal; the noise is then annealed to
  zero across fine-tuning.

Two ``init_kws`` worth knowing about:

* ``active_gp_thresh_ratio=0.0`` — keep *all* gene programs even if
  their measured activity is very low; ``> 0`` would prune low-activity
  programs after training.
* ``anndata_keys`` — override any of the 16 default ``storm_*`` AnnData
  key names. See :data:`storm.models.STORM_DEFAULT_KEYS`.

And one ``finetune_compile_kws`` knob:

* ``finetune_compile_kws`` — kwargs applied *only* to the fine-tune
  ``compile()`` call, on top of ``compile_kws``. Defaults bump
  ``lam_tmp`` and ``lam_align`` and shorten the per-modality burn-in
  (``n_epochs_rna = n_epochs_atac = 1``) so the fine-tune stage exits
  weight-freezing quickly; override here to suppress that behaviour.

In [6]:
print("==== Starting demo training (storm_tutorial.dill) ====")
storm_model = storm.models.fit_STORM(
    {"rna": rna, "atac": atac},
    guidance_hvf,
    model=storm.models.PairedSTORMModel,
    init_kws={"active_gp_thresh_ratio": 0.0},
    fit_kws={
        "directory": f"{MODEL_DIR}/glue_logs",
        "max_epochs": MAX_EPOCHS,
        "align_burnin": ALIGN_BURNIN,
        "safe_burnin": True,
        "patience": PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE,
    },
    compile_kws={
        "modality_weight": {"rna": 1.0, "atac": 3.0},
        "lr": 0.002,
        "lam_cos": 100.0,
        "lam_adj": 1000.0,
        "lam_masked_l1": 0.1,
        "lam_align": 1000.0,
        "lam_tmp": 10.0,
    },
)

storm_model.save(TUTORIAL_DILL)
print(f"\nSaved model to {TUTORIAL_DILL}")

==== Starting demo training (storm_tutorial.dill) ====
[INFO] fit_STORM: Pretraining STORM model...


INFO:fit_STORM:Pretraining STORM model...
/gpfs/gibbs/pi/zhao/xc384/data/STORM/storm/models/storm.py:1861: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /home/conda/feedstock_root/build_artifacts/pytorch-recipe_1696811120058/work/torch/csrc/utils/tensor_new.cpp:245.)
  gene_peaks_mask = torch.sparse_coo_tensor(


COSINE SIM GRAPH DECODER -> dropout_rate: 0.0
Number of vertices in STORMModel is: 6929
ONE HOP GCN NORM RNA NODE LABEL AGGREGATOR
MASKED TARGET RNA DECODER -> n_prior_gp_input: 2380, n_addon_gp_input: 0, n_output: 2871
MASKED SOURCE RNA DECODER -> n_prior_gp_input: 2380, n_addon_gp_input: 0, n_output: 2871
ONE HOP GCN NORM ATAC NODE LABEL AGGREGATOR
MASKED TARGET ATAC DECODER -> n_prior_gp_input: 2380, n_addon_gp_input: 0, n_output: 4058
MASKED SOURCE ATAC DECODER -> n_prior_gp_input: 2380, n_addon_gp_input: 0, n_output: 4058
[INFO] autodevice: Using CPU as computation device.


INFO:autodevice:Using CPU as computation device.


[INFO] check_graph: Checking variable coverage...


...

[INFO] EarlyStopping: No usable checkpoint found. Skipping checkpoint restoration.


INFO:EarlyStopping:No usable checkpoint found. Skipping checkpoint restoration.


[INFO] EarlyStopping: No usable checkpoint found. Skipping checkpoint restoration.


INFO:EarlyStopping:No usable checkpoint found. Skipping checkpoint restoration.
DEBUG:GraphDataset:Joined background process: 1665588



Saved model to ../dill_files/storm_tutorial.dill


## 6. Next steps

* [Tutorial 3 — clustering](./tutorial_3_clustering.ipynb) loads this
  dill, extracts the joint latent space (``X_storm``), and demonstrates
  the three clustering modes (modality-specific, joint).
* [Tutorial 4 — evaluation metrics](./tutorial_4_evaluation.ipynb)
  computes multi-omics integration (FOSCTTM, MLISI) and cross-modality
  consistency (ARI) metrics from the same artifacts.

Both downstream notebooks pick up the model from ``TRAINED_DILL`` —
point that at a longer-trained checkpoint to see publication-quality
figures.